In [1]:
# =========================================================
# FACTORIZATION MACHINE — PyTorch Version v2
# Features: 32 dims (bỏ utip_avg_comp theo kết quả MI)
#   USER(10) + BIZ(8) + TIP_USER(2) + TIP_BIZ(3)
#   + CHECKIN(1) + REVIEW(4) + TEMPORAL(4) = 32
# Fix vs v1:
#   - Bỏ utip_avg_comp (MI=0.0091, thấp nhất)
#   - lr=0.01 + CosineAnnealingWarmRestarts (thay ReduceLROnPlateau)
#   - min_delta=1e-6, patience=40 (nới early stopping)
# =========================================================

import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from datetime import datetime
import math
import random
import copy
import numpy as np
from collections import Counter

# =========================================================
# ID MAPPER
# =========================================================
class IDMapper:
    def __init__(self):
        self.map = {}

    def get(self, key):
        if key not in self.map:
            self.map[key] = len(self.map)
        return self.map[key]

# =========================================================
# BUILD CITY MAP (Top 50)
# =========================================================
def build_city_map(business_path, top_n=50):
    counter = Counter()
    with open(business_path, "r", encoding="utf-8") as f:
        for line in f:
            b    = json.loads(line)
            city = (b.get("city") or "Unknown").strip()
            counter[city] += 1
    top_cities = [city for city, _ in counter.most_common(top_n)]
    city_map   = {city: idx + 1 for idx, city in enumerate(top_cities)}
    print(f"  ✓ Built city map: top {top_n} cities (Other=0)")
    return city_map

# =========================================================
# LOAD USER CONTEXT  (10 features)
# =========================================================
def load_user_context(path):
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            u = json.loads(line)

            try:
                dt = datetime.strptime(
                    u.get("yelping_since", "2015-01-01 00:00:00"),
                    "%Y-%m-%d %H:%M:%S")
                years_active = max(0.0, (2020 - dt.year) / 10.0)
            except:
                years_active = 0.5

            elite_str = u.get("elite", "")
            elite_count = (len(elite_str) if isinstance(elite_str, list)
                           else len(elite_str.split(",")) if elite_str else 0)

            friends = u.get("friends", "")
            friend_count = (len([f for f in friends.split(",") if f.strip()])
                            if isinstance(friends, str) and friends and friends != "None"
                            else 0)

            compliment_keys = [
                "compliment_hot", "compliment_more", "compliment_profile",
                "compliment_cute", "compliment_list", "compliment_note",
                "compliment_plain", "compliment_cool", "compliment_funny",
                "compliment_writer", "compliment_photos"
            ]
            total_compliments = sum(u.get(k, 0) or 0 for k in compliment_keys)

            ctx[u["user_id"]] = [
                math.log1p(u.get("review_count", 0)) / 10.0,
                u.get("average_stars", 0) / 5.0,
                math.log1p(u.get("fans", 0)) / 10.0,
                math.log1p(max(0, u.get("useful", 0) or 0)) / 10.0,
                math.log1p(max(0, u.get("funny",  0) or 0)) / 10.0,
                math.log1p(max(0, u.get("cool",   0) or 0)) / 10.0,
                years_active,
                math.log1p(elite_count) / 10.0,
                math.log1p(friend_count) / 10.0,
                math.log1p(total_compliments) / 10.0,
            ]
    print(f"  ✓ Users: {len(ctx):,}")
    return ctx

# =========================================================
# LOAD BUSINESS CONTEXT  (8 features)
# =========================================================
def load_business_context(path, city_map):
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            b          = json.loads(line)
            categories = b.get("categories", "") or ""
            cat_count  = len(categories.split(",")) if categories else 0

            hours       = b.get("hours", {}) or {}
            open_days   = len(hours)
            total_hours = 0.0
            for day, hrs in hours.items():
                if not hrs or hrs == "0:0-0:0":
                    continue
                try:
                    open_t, close_t = hrs.split("-")
                    oh, om = map(int, open_t.split(":"))
                    ch, cm = map(int, close_t.split(":"))
                    total_hours += max(0, (ch*60+cm) - (oh*60+om)) / 60.0
                except:
                    total_hours += 8.0

            attrs     = b.get("attributes", {}) or {}
            price_raw = attrs.get("RestaurantsPriceRange2", 0) or 0
            try:
                price_norm = float(price_raw) / 4.0
            except:
                price_norm = 0.0

            city      = (b.get("city") or "Unknown").strip()
            city_norm = city_map.get(city, 0) / 50.0

            ctx[b["business_id"]] = [
                b.get("stars", 0) / 5.0,
                math.log1p(b.get("review_count", 0)) / 10.0,
                float(b.get("is_open", 0)),
                math.log1p(cat_count) / 10.0,
                open_days / 7.0,
                price_norm,
                min(total_hours, 168.0) / 168.0,
                city_norm,
            ]
    print(f"  ✓ Businesses: {len(ctx):,}")
    return ctx

# =========================================================
# LOAD CHECKIN CONTEXT  (1 feature)
# =========================================================
def load_checkin_context(path):
    ctx = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            c     = json.loads(line)
            bid   = c["business_id"]
            dates = c.get("date", "")
            ctx[bid] = math.log1p(len(dates.split(", ")) if dates else 0) / 10.0
    print(f"  ✓ Checkin: {len(ctx):,} businesses")
    return ctx

# =========================================================
# LOAD TIP CONTEXT
# =========================================================
def load_tip_context(path):
    user_tip = {}
    biz_tip  = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t      = json.loads(line)
            uid    = t["user_id"]
            bid    = t["business_id"]
            length = len(t.get("text", ""))
            comp   = t.get("compliment_count", 0)

            if uid not in user_tip:
                user_tip[uid] = [0, 0, 0]
            user_tip[uid][0] += 1
            user_tip[uid][1] += length
            user_tip[uid][2] += comp

            if bid not in biz_tip:
                biz_tip[bid] = [0, 0, 0]
            biz_tip[bid][0] += 1
            biz_tip[bid][1] += length
            biz_tip[bid][2] += comp

    print(f"  ✓ Tips: {len(user_tip):,} users, {len(biz_tip):,} businesses")
    return user_tip, biz_tip

# =========================================================
# FEATURE DIMS
# =========================================================
USER_CTX_DIM = 10
BIZ_CTX_DIM  = 8
TIP_USER_DIM = 2   # bỏ utip_avg_comp → còn utip_count, utip_avg_len
TIP_BIZ_DIM  = 3
CHECKIN_DIM  = 1
REVIEW_DIM   = 4
TEMPORAL_DIM = 4

NUM_DIM = (USER_CTX_DIM + BIZ_CTX_DIM +
           TIP_USER_DIM + TIP_BIZ_DIM +
           CHECKIN_DIM + REVIEW_DIM + TEMPORAL_DIM)  # = 32

FEATURE_NAMES = [
    # USER (10)
    "user_review_count", "user_avg_stars", "user_fans",
    "user_useful",       "user_funny",     "user_cool",
    "user_years_active", "user_elite",     "user_friends", "user_compliments",
    # BIZ (8)
    "biz_stars",       "biz_review_count", "biz_is_open",
    "biz_cat_count",   "biz_open_days",    "biz_price_range",
    "biz_total_hours", "biz_city",
    # TIP USER (2) — đã bỏ utip_avg_comp
    "utip_count", "utip_avg_len",
    # TIP BIZ (3)
    "btip_count", "btip_avg_len", "btip_avg_comp",
    # CHECKIN (1)
    "checkin_count",
    # REVIEW (4)
    "review_useful", "review_funny", "review_cool", "review_text_len",
    # TEMPORAL (4)
    "year", "month", "weekday", "hour",
]

assert len(FEATURE_NAMES) == NUM_DIM, \
    f"FEATURE_NAMES ({len(FEATURE_NAMES)}) != NUM_DIM ({NUM_DIM})"

print(f"NUM_DIM = {NUM_DIM}  (bỏ utip_avg_comp)")

# =========================================================
# DATASET
# =========================================================
class YelpDataset(IterableDataset):
    def __init__(self, review_path, user_ctx, biz_ctx,
                 user_tip, biz_tip, checkin_ctx, max_reviews=None):
        self.review_path = review_path
        self.user_ctx    = user_ctx
        self.biz_ctx     = biz_ctx
        self.user_tip    = user_tip
        self.biz_tip     = biz_tip
        self.checkin_ctx = checkin_ctx
        self.max_reviews = max_reviews
        self.user_map    = IDMapper()
        self.biz_map     = IDMapper()

    def __iter__(self):
        count = 0
        with open(self.review_path, "r", encoding="utf-8") as f:
            for line in f:
                if self.max_reviews and count >= self.max_reviews:
                    break

                r   = json.loads(line)
                uid = r["user_id"]
                bid = r["business_id"]

                uidx = self.user_map.get(uid)
                bidx = self.biz_map.get(bid)

                # USER (10)
                uctx = self.user_ctx.get(uid, [0.0] * USER_CTX_DIM)

                # BIZ (8)
                bctx = self.biz_ctx.get(bid, [0.0] * BIZ_CTX_DIM)

                # TIP USER (2) — bỏ utip_avg_comp
                utip = self.user_tip.get(uid, [0, 0, 0])
                utip_feat = [
                    math.log1p(utip[0]) / 10.0,                        # utip_count
                    math.log1p(utip[1] / max(utip[0], 1)) / 10.0,     # utip_avg_len
                    # utip_avg_comp đã bỏ
                ]

                # TIP BIZ (3)
                btip = self.biz_tip.get(bid, [0, 0, 0])
                btip_feat = [
                    math.log1p(btip[0]) / 10.0,
                    math.log1p(btip[1] / max(btip[0], 1)) / 10.0,
                    math.log1p(btip[2] / max(btip[0], 1)) / 10.0,
                ]

                # CHECKIN (1)
                checkin_feat = [self.checkin_ctx.get(bid, 0.0)]

                # REVIEW (4)
                useful   = max(0, r.get("useful", 0) or 0)
                funny    = max(0, r.get("funny",  0) or 0)
                cool     = max(0, r.get("cool",   0) or 0)
                text_len = len(r.get("text", "") or "")
                review_feat = [
                    math.log1p(useful)   / 10.0,
                    math.log1p(funny)    / 10.0,
                    math.log1p(cool)     / 10.0,
                    math.log1p(text_len) / 10.0,
                ]

                # TEMPORAL (4)
                dt = datetime.strptime(r["date"], "%Y-%m-%d %H:%M:%S")
                temporal_feat = [
                    (dt.year - 2015) / 10.0,
                    dt.month    / 12.0,
                    dt.weekday() / 6.0,
                    dt.hour     / 23.0,
                ]

                num_feat = torch.tensor(
                    uctx + bctx + utip_feat + btip_feat +
                    checkin_feat + review_feat + temporal_feat,
                    dtype=torch.float32
                )

                assert len(num_feat) == NUM_DIM, \
                    f"Dim mismatch: {len(num_feat)} != {NUM_DIM}"

                yield (
                    torch.tensor(uidx),
                    torch.tensor(bidx),
                    num_feat,
                    torch.tensor(r["stars"], dtype=torch.float32),
                )
                count += 1

# =========================================================
# FM MODEL
# =========================================================
class FMModel(nn.Module):
    def __init__(self, n_user, n_business, num_dim, k=128, dropout=0.15):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.k       = k
        self.num_dim = num_dim
        self.num_bn  = nn.BatchNorm1d(num_dim)

        self.bias     = nn.Parameter(torch.tensor([3.7840]))
        self.user_lin = nn.Embedding(n_user, 1)
        self.biz_lin  = nn.Embedding(n_business, 1)
        self.num_lin  = nn.Parameter(torch.zeros(num_dim))

        self.user_emb = nn.Embedding(n_user, k)
        self.biz_emb  = nn.Embedding(n_business, k)
        self.num_emb  = nn.Parameter(torch.zeros(num_dim, k))

        for param in [self.user_lin.weight, self.biz_lin.weight,
                      self.user_emb.weight, self.biz_emb.weight]:
            nn.init.normal_(param, 0, 0.01)
        nn.init.normal_(self.num_lin, 0, 0.01)
        nn.init.normal_(self.num_emb, 0, 0.01)

    def forward(self, user, biz, num):
        num    = self.num_bn(num)
        linear = (
            self.bias
            + self.user_lin(user).squeeze(-1)
            + self.biz_lin(biz).squeeze(-1)
            + (num * self.num_lin).sum(dim=1)
        )

        u       = self.user_emb(user).unsqueeze(1)
        b       = self.biz_emb(biz).unsqueeze(1)
        n       = num.unsqueeze(-1) * self.num_emb
        all_emb = torch.cat([u, b, n], dim=1)
        all_emb = self.dropout(all_emb)

        sum_sq = all_emb.sum(dim=1) ** 2
        sq_sum = (all_emb ** 2).sum(dim=1)
        inter  = 0.5 * (sum_sq - sq_sum).sum(dim=1)

        out = linear + inter
        if not self.training:
            out = torch.clamp(out, min=1.0, max=5.0)
        return out

# =========================================================
# METRICS
# =========================================================
def compute_metrics(predictions, targets):
    predictions = predictions.cpu()
    targets     = targets.cpu()
    mse    = torch.mean((predictions - targets) ** 2).item()
    rmse   = math.sqrt(mse)
    mae    = torch.mean(torch.abs(predictions - targets)).item()
    ss_res = torch.sum((targets - predictions) ** 2).item()
    ss_tot = torch.sum((targets - torch.mean(targets)) ** 2).item()
    r2     = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for user, biz, num, rating in loader:
            pred = model(user.to(device), biz.to(device), num.to(device))
            all_preds.append(pred.cpu())
            all_targets.append(rating)
    model.train()
    return compute_metrics(torch.cat(all_preds), torch.cat(all_targets))

def print_metrics(metrics, prefix=""):
    print(f"{prefix}MSE:{metrics['MSE']:.4f} | RMSE:{metrics['RMSE']:.4f} "
          f"| MAE:{metrics['MAE']:.4f} | R2:{metrics['R2']:.4f}")

# =========================================================
# PERMUTATION IMPORTANCE
# =========================================================
def permutation_importance(model, val_loader, device, n_repeats=3):
    model.eval()
    baseline = evaluate_model(model, val_loader, device)['RMSE']
    print(f"\nBaseline Val RMSE: {baseline:.4f}")
    print(f"{'Feature':<25} {'RMSE change':>12}  Đánh giá")
    print("-" * 55)

    results = []
    for feat_idx, feat_name in enumerate(FEATURE_NAMES):
        rmse_list = []
        for _ in range(n_repeats):
            all_preds, all_targets = [], []
            with torch.no_grad():
                for user, biz, num, rating in val_loader:
                    user  = user.to(device)
                    biz   = biz.to(device)
                    num   = num.to(device)
                    num_s = num.clone()
                    perm  = torch.randperm(num.shape[0])
                    num_s[:, feat_idx] = num[perm, feat_idx]
                    pred  = model(user, biz, num_s)
                    all_preds.append(pred.cpu())
                    all_targets.append(rating)
            rmse = compute_metrics(
                torch.cat(all_preds), torch.cat(all_targets))['RMSE']
            rmse_list.append(rmse - baseline)

        avg = float(np.mean(rmse_list))
        results.append((feat_name, avg))
        tag = ("✓ quan trọng" if avg > 0.005
               else "✗ gây nhiễu" if avg < -0.001
               else "~ trung tính")
        print(f"{feat_name:<25} {avg:>+.4f}       {tag}")

    results.sort(key=lambda x: x[1], reverse=True)
    return results

# =========================================================
# TRAIN
# =========================================================
def train_model(
    dataset,
    epochs                   = 120,
    batch_size               = 2048,
    train_split              = 0.8,
    val_split                = 0.1,
    test_split               = 0.1,
    early_stopping_patience  = 40,    # nới từ 30 → 40
    early_stopping_min_delta = 1e-6,  # nới từ 1e-5 → 1e-6
    eval_every               = 1,
    # Cosine schedule params
    lr                       = 0.01,
    T_0                      = 20,    # chu kỳ đầu (epochs)
    T_mult                   = 2,     # mỗi restart chu kỳ x2
    eta_min                  = 1e-5,  # lr tối thiểu
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    print(f"NUM_DIM = {NUM_DIM}")

    assert math.isclose(train_split + val_split + test_split, 1.0, rel_tol=1e-6)

    print("Loading data into memory...")
    all_data = list(dataset)
    random.shuffle(all_data)

    n     = len(all_data)
    n_tr  = int(n * train_split)
    n_val = int(n * val_split)

    train_data = all_data[:n_tr]
    val_data   = all_data[n_tr:n_tr + n_val]
    test_data  = all_data[n_tr + n_val:]
    print(f"Train:{len(train_data):,} | Val:{len(val_data):,} | Test:{len(test_data):,}")

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=batch_size, shuffle=False)

    n_user     = len(dataset.user_map.map) + 1000
    n_business = len(dataset.biz_map.map) + 1000
    print(f"Users:{n_user-1000:,} | Businesses:{n_business-1000:,}")

    model   = FMModel(n_user=n_user, n_business=n_business, num_dim=NUM_DIM).to(device)
    total_p = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_p:,}")

    optimizer = optim.SGD(
        model.parameters(),
        lr           = lr,
        momentum     = 0.9,
        weight_decay = 1e-5,
        nesterov     = True,
    )
    loss_fn = nn.MSELoss()

    # CosineAnnealingWarmRestarts:
    # - Bắt đầu với lr cao, giảm dần theo cosine trong T_0 epochs
    # - Sau T_0 epochs restart lại với lr cao, chu kỳ tiếp = T_0 * T_mult
    # - Tránh plateau, SGD tiếp tục học dù đã nhiều epochs
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0     = T_0,
        T_mult  = T_mult,
        eta_min = eta_min,
    )

    print(f"\nScheduler: CosineAnnealingWarmRestarts")
    print(f"  T_0={T_0} | T_mult={T_mult} | eta_min={eta_min}")
    print(f"  Chu kỳ: {T_0} → {T_0*T_mult} → {T_0*T_mult*T_mult} epochs")
    print(f"Early stopping: patience={early_stopping_patience} | min_delta={early_stopping_min_delta}")

    history = {
        'train_loss': [], 'val_mse': [], 'val_rmse': [],
        'val_mae': [], 'val_r2': [], 'lr': []
    }

    best_val_rmse = float('inf')
    best_state    = None
    best_epoch    = 0
    no_improve    = 0

    for ep in range(epochs):
        model.train()
        total_loss, total_count = 0.0, 0

        for user, biz, num, rating in train_loader:
            user   = user.to(device)
            biz    = biz.to(device)
            num    = num.to(device)
            rating = rating.to(device)

            if torch.isnan(num).any():
                continue

            pred = model(user, biz, num)
            loss = loss_fn(pred, rating)

            if torch.isnan(loss):
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # CosineAnnealing update theo step (không phải epoch)
            scheduler.step(ep + total_count / max(total_count + 1, 1))

            total_loss  += loss.item() * len(user)
            total_count += len(user)

        avg_loss   = total_loss / total_count if total_count > 0 else float('nan')
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(avg_loss)
        history['lr'].append(current_lr)

        if (ep + 1) % eval_every == 0:
            val_metrics = evaluate_model(model, val_loader, device)
            history['val_mse'].append(val_metrics['MSE'])
            history['val_rmse'].append(val_metrics['RMSE'])
            history['val_mae'].append(val_metrics['MAE'])
            history['val_r2'].append(val_metrics['R2'])

            print(f"Epoch {ep+1:3d}/{epochs} | Loss:{avg_loss:.4f} | LR:{current_lr:.6f}")
            print_metrics(val_metrics, prefix="  Val -> ")

            if val_metrics['RMSE'] < (best_val_rmse - early_stopping_min_delta):
                best_val_rmse = val_metrics['RMSE']
                best_state    = copy.deepcopy(model.state_dict())
                best_epoch    = ep + 1
                no_improve    = 0
                print(f"  >> New best RMSE: {best_val_rmse:.4f}")
            else:
                no_improve += 1
                print(f"  >> No improve {no_improve}/{early_stopping_patience}")
                if no_improve >= early_stopping_patience:
                    print(f"\nEarly stopping tại epoch {ep+1}")
                    history['stopped_epoch'] = ep + 1
                    break
        else:
            print(f"Epoch {ep+1:3d}/{epochs} | Loss:{avg_loss:.4f} | LR:{current_lr:.6f}")

    if best_state:
        model.load_state_dict(best_state)
        print(f"\nLoaded best model epoch {best_epoch} | RMSE:{best_val_rmse:.4f}")

    history['best_epoch']    = best_epoch
    history['best_val_rmse'] = best_val_rmse

    # Final evaluation
    print("\n" + "="*50)
    print("FINAL EVALUATION")
    print("="*50)
    train_m = evaluate_model(model, train_loader, device)
    val_m   = evaluate_model(model, val_loader,   device)
    test_m  = evaluate_model(model, test_loader,  device)
    print("\nTrain:");      print_metrics(train_m, "  ")
    print("\nValidation:"); print_metrics(val_m,   "  ")
    print("\nTest:");       print_metrics(test_m,  "  ")
    history['final_test_metrics'] = test_m

    # Permutation importance
    print("\n" + "="*50)
    print("PERMUTATION IMPORTANCE")
    print("="*50)
    perm_results = permutation_importance(model, val_loader, device)
    history['permutation_importance'] = perm_results

    return model, history, val_loader

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    BASE = "/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset"

    print("Loading contexts...")
    city_map             = build_city_map(
                               f"{BASE}/yelp_academic_dataset_business.json", top_n=50)
    user_ctx             = load_user_context(
                               f"{BASE}/yelp_academic_dataset_user.json")
    biz_ctx              = load_business_context(
                               f"{BASE}/yelp_academic_dataset_business.json", city_map)
    user_tip, biz_tip    = load_tip_context(
                               f"{BASE}/yelp_academic_dataset_tip.json")
    checkin_ctx          = load_checkin_context(
                               f"{BASE}/yelp_academic_dataset_checkin.json")

    dataset = YelpDataset(
        review_path = f"{BASE}/yelp_academic_dataset_review.json",
        user_ctx    = user_ctx,
        biz_ctx     = biz_ctx,
        user_tip    = user_tip,
        biz_tip     = biz_tip,
        checkin_ctx = checkin_ctx,
        max_reviews = None,
    )

    model, history, val_loader = train_model(
        dataset,
        epochs                   = 120,
        batch_size               = 2048,
        train_split              = 0.8,
        val_split                = 0.1,
        test_split               = 0.1,
        early_stopping_patience  = 40,
        early_stopping_min_delta = 1e-6,
        eval_every               = 1,
        lr                       = 0.01,
        T_0                      = 20,
        T_mult                   = 2,
        eta_min                  = 1e-5,
    )

    import os
    os.makedirs("/kaggle/working/models", exist_ok=True)
    torch.save(model.state_dict(), "/kaggle/working/models/fm_pytorch_v2.pt")
    print("Model saved to /kaggle/working/models/fm_pytorch_v2.pt")

NUM_DIM = 32  (bỏ utip_avg_comp)
Loading contexts...
  ✓ Built city map: top 50 cities (Other=0)
  ✓ Users: 1,987,897
  ✓ Businesses: 150,346
  ✓ Tips: 301,758 users, 106,193 businesses
  ✓ Checkin: 131,930 businesses
Using device: cuda
NUM_DIM = 32
Loading data into memory...
Train:5,592,224 | Val:699,028 | Test:699,028
Users:1,987,929 | Businesses:150,346
Parameters: 276,099,668

Scheduler: CosineAnnealingWarmRestarts
  T_0=20 | T_mult=2 | eta_min=1e-05
  Chu kỳ: 20 → 40 → 80 epochs
Early stopping: patience=40 | min_delta=1e-06
Epoch   1/120 | Loss:1.0862 | LR:0.009939
  Val -> MSE:1.0592 | RMSE:1.0292 | MAE:0.7730 | R2:0.5160
  >> New best RMSE: 1.0292
Epoch   2/120 | Loss:1.0784 | LR:0.009756
  Val -> MSE:1.0560 | RMSE:1.0276 | MAE:0.7703 | R2:0.5175
  >> New best RMSE: 1.0276
Epoch   3/120 | Loss:1.0754 | LR:0.009456
  Val -> MSE:1.0557 | RMSE:1.0275 | MAE:0.7702 | R2:0.5176
  >> New best RMSE: 1.0275
Epoch   4/120 | Loss:1.0724 | LR:0.009046
  Val -> MSE:1.0537 | RMSE:1.0265 | MA